<a href="https://colab.research.google.com/github/ivashenceva220495-beep/My_first_repository/blob/main/%D0%94%D0%97_PySpark_%D1%87%D0%B0%D1%81%D1%82%D1%8C2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Подключение к Pyspark (apt- get работает для linux)

In [ ]:
!apt-get update

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [ ]:
!ls

sample_data  spark-3.5.1-bin-hadoop3  spark-3.5.1-bin-hadoop3.tgz


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

Загружаем датасет титаника

In [ ]:
!wget https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv

--2026-02-17 12:13:47--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘titanic.csv’

titanic.csv         100%[===================>]  58.89K  --.-KB/s    in 0.03s   

2026-02-17 12:13:47 (1.98 MB/s) - ‘titanic.csv’ saved [60302/60302]



In [ ]:
df_iris= spark.read.csv('iris.CSV', inferSchema=True, header=True)
df_iris.show()
df_iris.printSchema()

+------------+-----------+------------+-----------+-------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|variety|variety_num|
+------------+-----------+------------+-----------+-------+-----------+
|         5.1|        3.5|         1.4|        0.2| Setosa|          0|
|         4.9|        3.0|         1.4|        0.2| Setosa|          0|
|         4.7|        3.2|         1.3|        0.2| Setosa|          0|
|         4.6|        3.1|         1.5|        0.2| Setosa|          0|
|         5.0|        3.6|         1.4|        0.2| Setosa|          0|
|         5.4|        3.9|         1.7|        0.4| Setosa|          0|
|         4.6|        3.4|         1.4|        0.3| Setosa|          0|
|         5.0|        3.4|         1.5|        0.2| Setosa|          0|
|         4.4|        2.9|         1.4|        0.2| Setosa|          0|
|         4.9|        3.1|         1.5|        0.1| Setosa|          0|
|         5.4|        3.7|         1.5|        0.2| Setosa|     

In [ ]:
df_iris.describe().show()

+-------+------------------+-------------------+------------------+------------------+---------+------------------+
|summary|      sepal_length|        sepal_width|      petal_length|       petal_width|  variety|       variety_num|
+-------+------------------+-------------------+------------------+------------------+---------+------------------+
|  count|               150|                150|               150|               150|      150|               150|
|   mean| 5.843333333333335|  3.057333333333334|3.7580000000000027| 1.199333333333334|     NULL|               1.0|
| stddev|0.8280661279778637|0.43586628493669793|1.7652982332594662|0.7622376689603467|     NULL|0.8192319205190406|
|    min|               4.3|                2.0|               1.0|               0.1|   Setosa|                 0|
|    max|               7.9|                4.4|               6.9|               2.5|Virginica|                 2|
+-------+------------------+-------------------+------------------+-----

In [ ]:
# Список признаков (все числовые колонки кроме целевой)
feature_columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

In [ ]:
from pyspark.ml.feature import VectorAssembler


In [ ]:
# Создаем VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol='features')

In [ ]:
# Трансформируем данные
df_with_features = assembler.transform(df_iris)

In [ ]:
print("Результат после VectorAssembler:")
df_with_features.select('features', 'variety', 'variety_num').show(5, truncate=False)
df_with_features.printSchema()

Результат после VectorAssembler:
+-----------------+-------+-----------+
|features         |variety|variety_num|
+-----------------+-------+-----------+
|[5.1,3.5,1.4,0.2]|Setosa |0          |
|[4.9,3.0,1.4,0.2]|Setosa |0          |
|[4.7,3.2,1.3,0.2]|Setosa |0          |
|[4.6,3.1,1.5,0.2]|Setosa |0          |
|[5.0,3.6,1.4,0.2]|Setosa |0          |
+-----------------+-------+-----------+
only showing top 5 rows

root
 |-- sepal_length: double (nullable = true)
 |-- sepal_width: double (nullable = true)
 |-- petal_length: double (nullable = true)
 |-- petal_width: double (nullable = true)
 |-- variety: string (nullable = true)
 |-- variety_num: integer (nullable = true)
 |-- features: vector (nullable = true)



С использованием pipeline

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
# Создаем Pipeline (можно добавить другие этапы)
pipeline = Pipeline(stages=[assembler])

# Обучаем Pipeline (хотя assembler не требует обучения)
pipeline_model = pipeline.fit(df_iris)

# Трансформируем данные
df_with_features = pipeline_model.transform(df_iris)

print("Результат после Pipeline с VectorAssembler:")
df_with_features.select('features', 'variety', 'variety_num').show(5, truncate=False)

Результат после Pipeline с VectorAssembler:
+-----------------+-------+-----------+
|features         |variety|variety_num|
+-----------------+-------+-----------+
|[5.1,3.5,1.4,0.2]|Setosa |0          |
|[4.9,3.0,1.4,0.2]|Setosa |0          |
|[4.7,3.2,1.3,0.2]|Setosa |0          |
|[4.6,3.1,1.5,0.2]|Setosa |0          |
|[5.0,3.6,1.4,0.2]|Setosa |0          |
+-----------------+-------+-----------+
only showing top 5 rows



Разбить данные на train and test

In [ ]:
train, test = df_with_features.randomSplit([0.8, 0.2], seed=12345)

In [ ]:
train.show()

+------------+-----------+------------+-----------+----------+-----------+-----------------+
|sepal_length|sepal_width|petal_length|petal_width|   variety|variety_num|         features|
+------------+-----------+------------+-----------+----------+-----------+-----------------+
|         4.3|        3.0|         1.1|        0.1|    Setosa|          0|[4.3,3.0,1.1,0.1]|
|         4.4|        2.9|         1.4|        0.2|    Setosa|          0|[4.4,2.9,1.4,0.2]|
|         4.4|        3.0|         1.3|        0.2|    Setosa|          0|[4.4,3.0,1.3,0.2]|
|         4.4|        3.2|         1.3|        0.2|    Setosa|          0|[4.4,3.2,1.3,0.2]|
|         4.5|        2.3|         1.3|        0.3|    Setosa|          0|[4.5,2.3,1.3,0.3]|
|         4.6|        3.1|         1.5|        0.2|    Setosa|          0|[4.6,3.1,1.5,0.2]|
|         4.6|        3.4|         1.4|        0.3|    Setosa|          0|[4.6,3.4,1.4,0.3]|
|         4.6|        3.6|         1.0|        0.2|    Setosa|        

Создадим и обучим модель логистической регрессии

In [ ]:
from pyspark.ml.classification import LogisticRegression

In [ ]:
lr = LogisticRegression(featuresCol = 'features', labelCol = 'variety_num')
lrModel = lr.fit(train)

In [ ]:
train_res = lrModel.transform(train)
test_res = lrModel.transform(test)

In [ ]:
train_res.show()

+------------+-----------+------------+-----------+----------+-----------+-----------------+--------------------+--------------------+----------+
|sepal_length|sepal_width|petal_length|petal_width|   variety|variety_num|         features|       rawPrediction|         probability|prediction|
+------------+-----------+------------+-----------+----------+-----------+-----------------+--------------------+--------------------+----------+
|         4.3|        3.0|         1.1|        0.1|    Setosa|          0|[4.3,3.0,1.1,0.1]|[67.7027919587531...|[1.0,1.6424718354...|       0.0|
|         4.4|        2.9|         1.4|        0.2|    Setosa|          0|[4.4,2.9,1.4,0.2]|[57.0087434717148...|[1.0,1.4036227809...|       0.0|
|         4.4|        3.0|         1.3|        0.2|    Setosa|          0|[4.4,3.0,1.3,0.2]|[61.7475610236625...|[1.0,2.4772179810...|       0.0|
|         4.4|        3.2|         1.3|        0.2|    Setosa|          0|[4.4,3.2,1.3,0.2]|[68.8060989728479...|[1.0,1.2455

Оценим качество

In [ ]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator


In [ ]:
ev = MulticlassClassificationEvaluator(labelCol='variety_num')

In [ ]:
ev.evaluate(train_res)

0.9844961240310077

In [ ]:
ev.evaluate(test_res)

1.0

Обучите модель дерева решений и оцените его качество

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

In [ ]:
tr = DecisionTreeClassifier(featuresCol='features', labelCol='variety_num')

In [ ]:
trFitted = tr.fit(train)

In [ ]:
train_tr_res=trFitted.transform(train)
test_tr_res=trFitted.transform(test)

In [ ]:
train_tr_res.show()

+------------+-----------+------------+-----------+----------+-----------+-----------------+--------------+-------------+----------+
|sepal_length|sepal_width|petal_length|petal_width|   variety|variety_num|         features| rawPrediction|  probability|prediction|
+------------+-----------+------------+-----------+----------+-----------+-----------------+--------------+-------------+----------+
|         4.3|        3.0|         1.1|        0.1|    Setosa|          0|[4.3,3.0,1.1,0.1]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|         4.4|        2.9|         1.4|        0.2|    Setosa|          0|[4.4,2.9,1.4,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|         4.4|        3.0|         1.3|        0.2|    Setosa|          0|[4.4,3.0,1.3,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|         4.4|        3.2|         1.3|        0.2|    Setosa|          0|[4.4,3.2,1.3,0.2]|[43.0,0.0,0.0]|[1.0,0.0,0.0]|       0.0|
|         4.5|        2.3|         1.3|        0.3|    Setosa|       

In [ ]:
ev.evaluate(train_tr_res)

0.9922428036123125

In [ ]:
ev.evaluate(test_tr_res)

1.0